In [10]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Dense, Dropout, Flatten
from tensorflow.keras.optimizers import Adam
import numpy as np
import matplotlib.pyplot as plt

In [11]:
# 1. DATA PREPROCESSING (AUGMENTATION)
# ================================
train_datagen = ImageDataGenerator(
    rescale=1./255,        # Normalize pixel values
    rotation_range=20,     # Rotate images randomly
    width_shift_range=0.2, # Shift images horizontally
    height_shift_range=0.2,# Shift images vertically
    shear_range=0.2,       # Shear transformation
    zoom_range=0.2,        # Zoom in/out
    horizontal_flip=True,  # Flip images horizontally
    fill_mode='nearest'    # Fill missing pixels
)

valid_datagen = ImageDataGenerator(rescale=1./255)  # Only rescale for validation
test_datagen = ImageDataGenerator(rescale=1./255)   # Only rescale for testing


In [12]:
train_data_gen = train_datagen.flow_from_directory(
    "chest_xray/train",
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'  # 3 classes: Normal, Bacterial, Viral
)

valid_data_gen = valid_datagen.flow_from_directory(
    "chest_xray/val",
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

test_data_gen = test_datagen.flow_from_directory(
    "chest_xray/test",
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

# Get class labels
class_labels = list(train_data_gen.class_indices.keys())


Found 5120 images belonging to 3 classes.
Found 490 images belonging to 3 classes.
Found 134 images belonging to 3 classes.


In [13]:
# 3. BUILD MODEL (VGG16 Transfer Learning)
# ================================
# Load VGG16 model with pretrained weights
base_model = VGG16(weights="imagenet", include_top=False, input_shape=(224, 224, 3))

# Freeze the base model layers (don't train them)
base_model.trainable = False

# Add custom layers on top
x = Flatten()(base_model.output)
x = Dense(1024, activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.3)(x)
output = Dense(3, activation='softmax')(x)  # 3 classes (Normal, Bacterial, Viral)

# Create the final model
model = Model(inputs=base_model.input, outputs=output)


In [14]:
# ================================
model.compile(optimizer=Adam(learning_rate=0.0001), loss='categorical_crossentropy', metrics=['accuracy'])


In [15]:
# 5. TRAIN THE MODEL
# ================================
nb_epochs = 10
nb_train_steps = len(train_data_gen)
nb_valid_steps = len(valid_data_gen)

In [ ]:

history = model.fit(
    train_data_gen,
    epochs=nb_epochs,
    steps_per_epoch=nb_train_steps,
    validation_data=valid_data_gen,
    validation_steps=nb_valid_steps
)



Epoch 1/10
 62/160 ━━━━━━━━━━━━━━━━━━━━ 14:31 9s/step - accuracy: 0.5037 - loss: 1.0958

In [ ]:
predictions = model.predict(test_generator)
predicted_classes = (predictions > 0.5).astype("int32")
